## Tablemate - Cell 1 - Importer och Table

In [32]:
import csv
from datetime import datetime

class Table: #skapar en klass Table som representerar en tabell för borden
    def __init__(
        self,
        table_id,
        table_capacity,
        table_placement,
        table_type,
        table_description,
        is_bookable):

        self.table_id = table_id
        self.table_capacity = table_capacity
        self.table_placement = table_placement
        self.table_type = table_type
        self.table_description = table_description
        self.is_bookable = is_bookable

    def display_information(self): #skapar en metod display_information som skriver ut information om tabellen
        return (
        f" Table: {self.table_id}, "
        f" Seats: {self.table_capacity}, "
        f" Placement: {self.table_placement}, "
        f" Type: {self.table_type}, "
        f" Description: {self.table_description}, "
        f" Bookable: {self.is_bookable}")

    def can_seat_guests(self, guest_count):
        return self.is_bookable and guest_count <= self.table_capacity

### Cell 2 - TerraceTable

In [33]:
class TerraceTable(Table): #skapar en barnklass med TerraceTable
    def display_information(self):
        base_information = super().display_information()

        return (
            f"{base_information}, "
            f"Outdoor Seating: Yes" )

### Cell 3 - load_tables()

In [ ]:
def load_tables(file_import): #skapar en funktion --load_tables-- som läser in tabeller från en CSV-fil
    tables = [] #skapar listan tables som kommer att innehålla alla tabeller

    try:
        with open(file_import, "r", encoding="utf-8", newline="") as file: #öppnar datafilen
            reader = csv.DictReader(file) #använder rubriker som nycklar för att läsa in data
            for row in reader: #loopar igenom ett bord i taget

                row["table_id"] = int(row["table_id"]) #konverterar till en int
                row["table_capacity"] = int(row["table_capacity"]) #konverterar  till en int
                row["is_bookable"] = row["is_bookable"].strip().lower() == "true" #konverterar  till en bool

                if row["table_type"].strip().lower() == "terrace": #
                    table = TerraceTable(
                        row["table_id"],
                        row["table_capacity"],
                        row["table_placement"],
                        row["table_type"],
                        row["table_description"],
                        row["is_bookable"])
               
                else:
                    table = Table(
                        row["table_id"],
                        row["table_capacity"],
                         row["table_placement"],
                        row["table_type"],
                        row["table_description"],
                        row["is_bookable"])

                tables.append(table) #Alla bord läggs till oavsett typ.

        return tables

    except FileNotFoundError: #om filen inte hittas
        print(f"Error: The file '{file_import}' was not found.") #skriver ut ett felmeddelande

    except (ValueError, KeyError) as error: #om det finns ett värdefel eller nyckelfel
        print(f"Error while reading '{file_import}': {error}") #skriver ut ett felmeddelande

    except csv.Error as error: #om det finns ett csv-fel
        print(f"Error: An error occurred while processing '{file_import}': {error}") #skriver ut ett felmeddelande
    return []

### Cell 4 - load_bookings()

In [35]:
#Läsa bokningar
def load_bookings(file_import):
    bookings = []

    try:
        with open(file_import,"r",encoding="utf-8", newline="") as file:
            reader = csv.DictReader(file)

            for row in reader:
                row["booking_id"] = int(row["booking_id"])
                row["guest_count"] = int(row["guest_count"])
                row["table_id"] = int(row["table_id"])

                bookings.append(row)

        return bookings

    except FileNotFoundError:
        print(f"Error: The file '{file_import}' was not found.") #skriver ut ett felmeddelande

    except (ValueError, KeyError) as error: #om det finns ett värdefel eller nyckelfel
        print(f"Error while reading '{file_import}': {error}") #skriver ut ett felmeddelande

    except csv.Error as error: #om det finns ett csv-fel
        print(f"Error: An error occurred while processing '{file_import}': {error}") #skriver ut ett felmeddelande

    return []

### Cell 5 - is_table_available()

In [40]:
def is_table_available(
        bookings,
        table_id,
        booking_date,
        booking_time):

    for booking in bookings:
        same_table = booking["table_id"] == table_id
        same_date = booking["booking_date"] == booking_date
        same_time = booking["booking_time"] == booking_time
        is_active = (booking["booking_status"].lower() == "active")

        if (
            same_table
            and same_date
            and same_time
            and is_active):

            return False

    return True

### Cell 6 - bokningsfält och save_booking()

In [38]:
BOOKING_FIELDS = [
    "booking_id",
    "customer_name",
    "booking_date",
    "booking_time",
    "guest_count",
    "table_id",
    "booking_status",
    "created_at",]

def save_booking(file_import, booking):
    try:
        with open(file_import, "a", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file,fieldnames=BOOKING_FIELDS)

            if file.tell() == 0:
                writer.writeheader()
            writer.writerow(booking)

        return True

    except (OSError, csv.Error) as error:
        print(f"Error while saving the booking: {error}")

        return False

### Cell 7 - create_booking()

In [ ]:
def create_booking(
        bookings,
        table,
        customer_name,
        booking_date,
        booking_time,
        guest_count,
        file_import):

        if not table.can_seat_guests(guest_count):
            print( "The table cannot accommodate the requested amount of guests.")

            return None

        if not is_table_available(
              bookings,
              table.table_id,
              booking_date,
              booking_time):
            print("The table is already booked at the selected time and date.")

            return None
        
        new_booking_id = 1

        for existing_booking in bookings:
             if (existing_booking["booking_id"] >= new_booking_id):
  
                  new_booking_id = (existing_booking["booking_id"] + 1)

        new_booking = {
             "booking_id": new_booking_id,
             "customer_name": customer_name,
             "booking_date": booking_date,
             "booking_time": booking_time,
             "guest_count": guest_count,
             "table_id": table.table_id,
             "booking_status": "active",
             "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

        if save_booking(file_import, new_booking):
             bookings.append(new_booking)

             print(f"Booking {new_booking_id} was created for table {table.table_id}." )

             return new_booking

        return None

### Cell 8 - save_all_bookings()

In [46]:
#Uppdatera bokningsfilen

def save_all_bookings(file_import, bookings):
    try:
        with open(file_import, "w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=BOOKING_FIELDS)

            writer.writeheader()
            writer.writerows(bookings)

        return True

    except (OSError, csv.Error) as error:
        print(f"Error while updating bookings: {error}")

        return False

### Cell 9 - cancel_booking()

In [52]:
def cancel_booking(bookings, booking_id,file_import):

    for booking in bookings:
        correct_id = (
            booking["booking_id"]) == booking_id

        is_active = (
            booking["booking_status"].strip().lower() == "active")

        if correct_id and is_active:
            booking["booking_status"] = "cancelled"

            if save_all_bookings(file_import, bookings):
                print(f"Booking {booking_id} was cancelled.")

                return True

            booking["booking_status"] = "active"

            return False

    print(f"No active booking with ID {booking_id} was found.")

    return False

### Cell 10 - find_available_tables()

In [58]:
def find_available_tables(tables, bookings, guest_count, booking_date, booking_time):

    available_tables = []

    for table in tables:
        has_capacity = table.can_seat_guests(guest_count)

        is_available = is_table_available(
            bookings,
            table.table_id,
            booking_date,
            booking_time)

        if has_capacity and is_available:
            available_tables.append(table)

    return available_tables